# SageMaker Real-Time Model Deployment

Deploy trained medical imaging model to SageMaker real-time endpoint for synchronous inference.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '../../..'))
from utils import get_or_create_role

role = get_or_create_role()
print(f"SageMaker role: {role}")


In [ ]:
%pip install sagemaker boto3

In [ ]:
import sagemaker
import boto3
from sagemaker.pytorch import PyTorchModel
from sagemaker.predictor import Predictor
import json

In [ ]:
# Model configuration
sess = sagemaker.Session()
bucket = sess.default_bucket()
# Replace with your training job's model artifact path
model_data = f"s3://{bucket}/YOUR_TRAINING_JOB/output/model.tar.gz"
image_uri = "763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:2.5.1-gpu-py311-cu124-ubuntu22.04-sagemaker"

In [ ]:
endpoint_name = "medical-imaging-realtime-endpoint"
sm_client = boto3.client("sagemaker")
existing_endpoints = sm_client.list_endpoints()["Endpoints"]

if any(ep["EndpointName"] == endpoint_name for ep in existing_endpoints):
    print(f"Endpoint '{endpoint_name}' already exists. Skipping deployment.")
else:
    pytorch_model = PyTorchModel(
        model_data=model_data,
        role=role,
        source_dir=".",
        entry_point="inference_realtime.py",
        framework_version="2.5.1",
        py_version="py311",
        image_uri=image_uri,
        dependencies=["requirements.txt"]
    )
    predictor = pytorch_model.deploy(
        instance_type="ml.g5.xlarge",
        initial_instance_count=1,
        endpoint_name=endpoint_name
    )
    print(f"Endpoint '{endpoint_name}' deployed.")

In [ ]:
class MedicalImagingPredictor:
    def __init__(self, endpoint_name):
        self.endpoint_name = endpoint_name
        self.runtime_client = boto3.client('sagemaker-runtime')
        self.labels = [
            'Surgical_implant', 'Vertebral_collapse', 'Spondylolysthesis',
            'No_finding', 'Foraminal_stenosis', 'Other_lesions',
            'Disc_space_narrowing', 'Osteophytes'
        ]
    
    def predict(self, s3_image_uri):
        payload = json.dumps({"file_path": s3_image_uri})
        response = self.runtime_client.invoke_endpoint(
            EndpointName=self.endpoint_name,
            ContentType="application/json",
            Body=payload
        )
        return self._parse_response(response)
    
    def predict_local(self, local_path):
        with open(local_path, 'rb') as f:
            image_bytes = f.read()
        response = self.runtime_client.invoke_endpoint(
            EndpointName=self.endpoint_name,
            ContentType="application/dicom",
            Body=image_bytes
        )
        return self._parse_response(response)
    
    def _parse_response(self, response):
        result = json.loads(response['Body'].read().decode('utf-8'))
        predictions = result["predictions"][0]
        predicted_idx = result["predicted_class"][0]
        return {
            "probabilities": dict(zip(self.labels, predictions)),
            "predicted_class": self.labels[predicted_idx],
            "confidence": result["confidence"][0]
        }

In [ ]:
# Upload local sample to S3 for testing
s3_client = boto3.client('s3')
sample_key = "realtime-inference/samples/sample_image.dcm"
s3_client.upload_file("samples/sample_image.dcm", bucket, sample_key)
s3_image_uri = f"s3://{bucket}/{sample_key}"

# Test real-time inference
predictor_wrapper = MedicalImagingPredictor(endpoint_name)
result = predictor_wrapper.predict(s3_image_uri)

print(f"Predicted: {result['predicted_class']}")
print(f"Confidence: {result['confidence']:.4f}")
print("\nAll probabilities:")
for label, prob in result['probabilities'].items():
    print(f"  {label}: {prob:.4f}")

In [ ]:
# Test real-time inference with a local file (no S3 upload)
local_file = "samples/sample_image.dcm"
result = predictor_wrapper.predict_local(local_file)

print(f"Predicted: {result['predicted_class']}")
print(f"Confidence: {result['confidence']:.4f}")
print("\nAll probabilities:")
for label, prob in result['probabilities'].items():
    print(f"  {label}: {prob:.4f}")

In [ ]:
# Cleanup
endpoint_name = "medical-imaging-realtime-endpoint"
sm_client = boto3.client('sagemaker')

try:
    desc = sm_client.describe_endpoint(EndpointName=endpoint_name)
    endpoint_config_name = desc['EndpointConfigName']
    config_desc = sm_client.describe_endpoint_config(EndpointConfigName=endpoint_config_name)
    model_name = config_desc['ProductionVariants'][0]['ModelName']

    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f"Deleted endpoint: {endpoint_name}")

    sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    print(f"Deleted endpoint config: {endpoint_config_name}")

    sm_client.delete_model(ModelName=model_name)
    print(f"Deleted model: {model_name}")
except sm_client.exceptions.ClientError as e:
    print(f"Cleanup error: {e}")